# Optimización definitiva de Crop7

Optuna ajusta gamma, delta, bias y slope con estudios separados para acc_both y SSIM en Fashion-MNIST y MNIST. Se usan los mismos CVAE del cuaderno de Crop6, con dimensión intermedia 256 y dimensión latente 64.

Cada estudio usa 100 pruebas de 30 iteraciones sobre un lote fijo de 1.000 mezclas de entrenamiento, con semilla 3333. Los rangos de búsqueda son gamma y delta de 0 a 2, bias de 0.05 a 0.45 y slope de 5 a 60 en escala logarítmica.

Los estudios se guardan en `results_crop7/<dataset>/<objetivo>/Crop7.sqlite3`. Para retomar un estudio, volvé a ejecutar su celda y Optuna completará las pruebas pendientes.

## Configuración y evaluación

In [1]:
from pathlib import Path
import sys

import numpy as np
import optuna

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "project" / "CROP" / "models").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abrí el notebook desde el repositorio.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENTS_DIR = PROJECT_ROOT / "project" / "definitive_experiments"
RESULTS_DIR = EXPERIMENTS_DIR / "results_crop7"
SEED = 3333
LAT, INTER = 64, 256
N_IMAGES = 1000
N_TRIALS = 100
STUDY_ITERATIONS = 30

DATASETS = {"fashion": "Fashion-MNIST", "mnist": "MNIST"}
SEARCH_SPACE = {
    "gamma": [0.0, 2.0, False],
    "delta": [0.0, 2.0, False],
    "bias": [0.05, 0.45, False],
    "slope": [5.0, 60.0, True],
}
FIXED_PARAMS = {
    "alpha_1": -2,
    "alpha_2": -22,
    "alpha_mix": 0.5,
    "beta": 1,
}


def result_dir(dataset, objective_metric):
    return RESULTS_DIR / dataset / objective_metric


def load_inputs(dataset, latent_dim, intermediate_dim):
    import tensorflow as tf
    from project.CROP.models.crop7 import Crop7
    from project.loader import Loader

    tf.keras.utils.set_random_seed(SEED)
    return (
        Loader.data(dataset),
        Loader.cvae(model_name=f"int_{intermediate_dim}_lat_{latent_dim}_{dataset}.keras"),
        Loader.predictor(dataset=dataset),
        Crop7,
    )


def sample_batch(data, split, seed, n_images):
    rng = np.random.default_rng(seed)
    x0, x1 = data[f"x_{split}"], data[f"x_{split}_1"]
    y0, y1 = data[f"y_{split}"], data[f"y_{split}_1"]
    idx0 = rng.choice(len(x0), size=n_images, replace=False)
    idx1 = rng.choice(len(x1), size=n_images, replace=False)
    return x0[idx0], x1[idx1], y0[idx0], y1[idx1]


def evaluate(inputs, batch, params, seed, iterations):
    import tensorflow as tf

    data, cvae, predictor, model_class = inputs
    tf.keras.utils.set_random_seed(seed)
    model = model_class(cvae=cvae, predictor=predictor)
    return model.unmix(
        *batch,
        params=params,
        iterations=iterations,
        labels=data["labels"],
        show_metrics=False,
    )


def scalar_metric(metrics, key):
    value = metrics[key]
    return float(value[0] if isinstance(value, (tuple, list)) else value)

/home/galsan/venvs/tensorflow-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Estudios de Optuna

In [2]:
def study_location(dataset, objective_metric):
    path = result_dir(dataset, objective_metric) / "Crop7.sqlite3"
    name = f"{dataset}_{objective_metric}_Crop7"
    return path, f"sqlite:///{path.as_posix()}", name


def load_study(dataset, objective_metric):
    path, storage, name = study_location(dataset, objective_metric)
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el estudio en {path}")
    return optuna.load_study(storage=storage, study_name=name)


def optimize_crop7(dataset, objective_metric, inputs, train_batch):
    path, storage, name = study_location(dataset, objective_metric)
    path.parent.mkdir(parents=True, exist_ok=True)
    study = optuna.create_study(
        storage=storage,
        study_name=name,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        load_if_exists=True,
    )
    config = {
        "seed": SEED,
        "images": N_IMAGES,
        "iterations": STUDY_ITERATIONS,
        "latent_dim": LAT,
        "intermediate_dim": INTER,
        "fixed_params": FIXED_PARAMS,
        "search_space": SEARCH_SPACE,
    }
    saved_config = study.user_attrs.get("config")
    if saved_config is None and not study.get_trials(deepcopy=False):
        study.set_user_attr("config", config)
    elif saved_config != config:
        raise ValueError(f"La configuración guardada en {path} difiere de la actual.")

    def objective(trial):
        params = {
            key: trial.suggest_float(key, low, high, log=log)
            for key, (low, high, log) in SEARCH_SPACE.items()
        }
        effective_params = {**FIXED_PARAMS, **params}
        metrics = evaluate(inputs, train_batch, effective_params, SEED, STUDY_ITERATIONS)
        score = scalar_metric(metrics, objective_metric)
        if not np.isfinite(score):
            raise ValueError(f"Métrica no finita en {name}: {score}")
        trial.set_user_attr("effective_params", effective_params)
        for key in ("acc_both", "ssim", "recon_bpsnr"):
            value = scalar_metric(metrics, key)
            trial.set_user_attr(key, value if np.isfinite(value) else None)
        return score

    completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,), deepcopy=False)
    remaining = max(0, N_TRIALS - len(completed))
    if remaining:
        if completed:
            # Evitar repetir las muestras iniciales al retomar el estudio.
            study.sampler = optuna.samplers.TPESampler(seed=SEED + len(completed))
        study.optimize(objective, n_trials=remaining, n_jobs=1)

    print(f"{name} · {study.best_value:.6g} · {study.best_params}", flush=True)
    return study


def plot_study(study, dataset, objective_metric):
    from optuna.visualization import plot_slice

    fig = plot_slice(study, params=list(SEARCH_SPACE),
                     target_name=objective_metric)
    fig.update_layout(title=f"{DATASETS[dataset]} · Crop7 · {objective_metric}")
    fig.show()


def run_experiment(dataset, objective_metric):
    inputs = load_inputs(dataset, LAT, INTER)
    train_batch = sample_batch(inputs[0], "train", SEED, N_IMAGES)
    return optimize_crop7(dataset, objective_metric, inputs, train_batch)

## Fashion-MNIST

In [3]:
fashion_acc = run_experiment("fashion", "acc_both")

I0000 00:00:1795844742.384487  177162 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795844742.489760  177162 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795844744.503923  177162 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Usando fashion como dataset


I0000 00:00:1795844746.551614  177162 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1297 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
[I 2026-11-28 02:45:48,414] A new study created in RDB with name: fashion_acc_both_Crop7
I0000 00:00:1795844749.034512  177162 cuda_dnn.cc:461] Loaded cuDNN version 92801
[I 2026-11-28 02:45:52,404] Trial 0 finished with value: 0.432 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547, 'bias': 0.2444485116229741, 'slope': 17.313243515889607}. Best is trial 0 with value: 0.432.
[I 2026-11-28 02:45:53,351] Trial 1 finished with value: 0.321 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138, 'bias': 0.29511795626423765, 'slope': 6.7312641583709825}. Best is trial 0 with value: 0.432.
[I 2026-11-28 02:45:54,885] Trial 2 finished with value: 0.351 and parameters: {'gamma': 1.4244045932667562, 'delta': 0.7451210

fashion_acc_both_Crop7 · 0.667 · {'gamma': 0.13335985841516373, 'delta': 0.18813391489323197, 'bias': 0.23348548291435134, 'slope': 18.632003312976767}


In [4]:
fashion_ssim = run_experiment("fashion", "ssim")

Usando fashion como dataset


[I 2026-11-28 02:48:09,609] A new study created in RDB with name: fashion_ssim_Crop7
[I 2026-11-28 02:48:11,077] Trial 0 finished with value: 0.6375144720077515 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547, 'bias': 0.2444485116229741, 'slope': 17.313243515889607}. Best is trial 0 with value: 0.6375144720077515.
[I 2026-11-28 02:48:12,472] Trial 1 finished with value: 0.57804274559021 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138, 'bias': 0.29511795626423765, 'slope': 6.7312641583709825}. Best is trial 0 with value: 0.6375144720077515.
[I 2026-11-28 02:48:13,989] Trial 2 finished with value: 0.5782333612442017 and parameters: {'gamma': 1.4244045932667562, 'delta': 0.7451210841006557, 'bias': 0.259907309133894, 'slope': 9.618436608383888}. Best is trial 0 with value: 0.6375144720077515.
[I 2026-11-28 02:48:15,520] Trial 3 finished with value: 0.5554630756378174 and parameters: {'gamma': 1.2432624452089451, 'delta': 0.8981995207171

fashion_ssim_Crop7 · 0.661666 · {'gamma': 0.9634046740121409, 'delta': 0.17195513564097872, 'bias': 0.22840839778229388, 'slope': 24.03625120383506}


## MNIST

In [5]:
mnist_acc = run_experiment("mnist", "acc_both")

Usando mnist como dataset


[I 2026-11-28 02:50:32,887] A new study created in RDB with name: mnist_acc_both_Crop7
[I 2026-11-28 02:50:34,497] Trial 0 finished with value: 0.77 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547, 'bias': 0.2444485116229741, 'slope': 17.313243515889607}. Best is trial 0 with value: 0.77.
[I 2026-11-28 02:50:36,007] Trial 1 finished with value: 0.691 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138, 'bias': 0.29511795626423765, 'slope': 6.7312641583709825}. Best is trial 0 with value: 0.77.
[I 2026-11-28 02:50:37,380] Trial 2 finished with value: 0.653 and parameters: {'gamma': 1.4244045932667562, 'delta': 0.7451210841006557, 'bias': 0.259907309133894, 'slope': 9.618436608383888}. Best is trial 0 with value: 0.77.
[I 2026-11-28 02:50:38,929] Trial 3 finished with value: 0.531 and parameters: {'gamma': 1.2432624452089451, 'delta': 0.8981995207171292, 'bias': 0.13762544493463805, 'slope': 41.14490253969007}. Best is trial 0 with value: 

mnist_acc_both_Crop7 · 0.901 · {'gamma': 0.34888095935622965, 'delta': 0.3144714673112621, 'bias': 0.2214258524312783, 'slope': 35.57696668471005}


In [6]:
mnist_ssim = run_experiment("mnist", "ssim")

Usando mnist como dataset


[I 2026-11-28 02:52:58,123] A new study created in RDB with name: mnist_ssim_Crop7
[I 2026-11-28 02:52:59,724] Trial 0 finished with value: 0.7679494619369507 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547, 'bias': 0.2444485116229741, 'slope': 17.313243515889607}. Best is trial 0 with value: 0.7679494619369507.
[I 2026-11-28 02:53:01,194] Trial 1 finished with value: 0.6913569569587708 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138, 'bias': 0.29511795626423765, 'slope': 6.7312641583709825}. Best is trial 0 with value: 0.7679494619369507.
[I 2026-11-28 02:53:02,719] Trial 2 finished with value: 0.6704530715942383 and parameters: {'gamma': 1.4244045932667562, 'delta': 0.7451210841006557, 'bias': 0.259907309133894, 'slope': 9.618436608383888}. Best is trial 0 with value: 0.7679494619369507.
[I 2026-11-28 02:53:04,209] Trial 3 finished with value: 0.6140226125717163 and parameters: {'gamma': 1.2432624452089451, 'delta': 0.8981995207171

mnist_ssim_Crop7 · 0.819544 · {'gamma': 0.6116313569243619, 'delta': 0.09836984105115631, 'bias': 0.20321528539702086, 'slope': 34.791919805672556}


## Resultados guardados y evaluación en prueba

Elegí un estudio guardado para ver sus mejores parámetros y los gráficos de cortes de Optuna. La última celda evalúa esos parámetros en un lote independiente de prueba con la misma cantidad de mezclas e iteraciones del estudio.

In [10]:
dataset, objective_metric = "fashion", "ssim"

study = load_study(dataset, objective_metric)
print(f"Crop7 · {study.best_value:.6g} · {study.best_params}")
plot_study(study, dataset, objective_metric)

Crop7 · 0.661666 · {'gamma': 0.9634046740121409, 'delta': 0.17195513564097872, 'bias': 0.22840839778229388, 'slope': 24.03625120383506}


In [8]:
config = study.user_attrs["config"]
test_seed = config["seed"] + 1
inputs = load_inputs(dataset, config["latent_dim"], config["intermediate_dim"])
test_batch = sample_batch(inputs[0], "test", test_seed, config["images"])
params = {**config["fixed_params"], **study.best_params}
test_metrics = evaluate(inputs, test_batch, params, test_seed, config["iterations"])

for metric in ("acc_both", "ssim", "recon_bpsnr"):
    print(f"{metric} = {scalar_metric(test_metrics, metric):.6g}")

Usando fashion como dataset
acc_both = 0.548
ssim = 0.659333
recon_bpsnr = 17.9745
